In [ ]:
#Install dependencies
!pip install -q -U langchain langchain-community langchain-openai faiss-cpu openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#Set paths and API key
import os
from pathlib import Path


BASE_INDEX_DIR = Path('/content/drive/MyDrive/Additional_NPJ_Data/Doc_store')
OUTPUT_DIR = Path('/content/drive/MyDrive/NPJ_RAG/zero_shot_extractions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Option A: paste key here for quick testing. Do not share notebooks with your key inside.
os.environ['OPENAI_API_KEY'] = 'Your API Key'

# Option B: if you stored it as a Colab secret, uncomment this instead:
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

print('Base index directory:', BASE_INDEX_DIR)
print('Output directory:', OUTPUT_DIR)


Base index directory: /content/drive/MyDrive/Additional_NPJ_Data/Doc_store
Output directory: /content/drive/MyDrive/NPJ_RAG/zero_shot_extractions


In [ ]:
#Imports and configuration
import math
from typing import List, Optional

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

EMBEDDING_MODEL = 'text-embedding-ada-002'  # same as your stored index
LLM_MODEL = 'gpt-4o'

EXTRACTION_QUERY = '''
Please extract information from the attached document using the following categories:
Methods, Population (Inclusion/Exclusion), Intervention vs. Comparator, Outcomes,
and Final Conclusion.

For the results, provide a json including Outcome Type, Subgroup,
Intervention/Control Rates, and Comparison Statistics.

Stats/results only should be in JSON. Everything else should be plain text.
'''


/tmp/ipykernel_3873/1621950601.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
#Load your saved FAISS store
def load_vector_store(file_name: str, base_dir: Path = BASE_INDEX_DIR) -> FAISS:

    index_folder = base_dir / f'{file_name}'

    if not index_folder.exists():
        raise FileNotFoundError(f'Could not find FAISS index folder: {index_folder}')

    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

    vector_store = FAISS.load_local(
        str(index_folder),
        embeddings,
        allow_dangerous_deserialization=True
    )

    chunks = vector_store.index.ntotal
    print(f'Loaded: {index_folder}')
    print(f'Number of chunks in VDB: {chunks}')

    return vector_store


In [ ]:
#Retrieval, context building, and prompt construction
def retrieve_relevant_docs(
    vector_store: FAISS,
    query: str,
    k: Optional[int] = None
) -> List[Document]:

    total_chunks = vector_store.index.ntotal

    if k is None:
        k = math.ceil(total_chunks / 1)

    k = min(k, total_chunks)

    retriever = vector_store.as_retriever(search_kwargs={'k': k})
    docs = retriever.invoke(query)

    print(f'Retrieved chunks: {len(docs)}')
    return docs


def build_context(docs: List[Document], max_chars: int = 120_000) -> str:
    '''
    Builds context from retrieved chunks.
    Your existing FAISS.from_texts(...) may not store page metadata unless you added it.
    '''
    parts = []
    current_len = 0

    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get('page_num', doc.metadata.get('page', 'unknown'))
        source = doc.metadata.get('source', 'unknown')
        chunk_text = doc.page_content.strip()

        block = f'\n\n[Chunk {i} | source={source} | page={page}]\n{chunk_text}'

        if current_len + len(block) > max_chars:
            print(f'Context truncated at chunk {i} due to max_chars={max_chars}')
            break

        parts.append(block)
        current_len += len(block)

    print(f'Context characters: {current_len}')
    return '\n'.join(parts)


def build_extraction_prompt(context: str) -> str:
    return f'''
You are extracting information from a clinical or research document.

Use only the document context provided below.
Do not use outside knowledge.
Do not invent information.
If information is missing, write "Not reported".

Follow this formatting rule strictly:
- Methods (Design, Allocation concealment, Blinding, Follow-up duration, Setting, Sample size), Population, Intervention vs Comparator, Outcomes, Patient Follow-up, Intention to Treat Analysis, Baseline Characteristics, Sources of funding
 and Final Conclusion must be plain text.
- Only Stats JSON should be JSON.
- Do not put the full answer inside JSON.
- Keep the answer simple.

Original user request:
{EXTRACTION_QUERY}

Document context:
{context}

'''


In [ ]:
#LLM call and full extraction pipeline
def ask_llm(prompt: str) -> str:
    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0.5
    )
    response = llm.invoke(prompt)
    return response.content


def extract_from_index(
    file_name: str,
    k: Optional[int] = None,
    max_chars: int = 120_000,
    save_output: bool = True
) -> str:

    vector_store = load_vector_store(file_name)

    docs = retrieve_relevant_docs(
        vector_store=vector_store,
        query=EXTRACTION_QUERY,
        k=k
    )

    if not docs:
        raise RuntimeError('No documents retrieved from FAISS index.')

    context = build_context(docs, max_chars=max_chars)
    final_prompt = build_extraction_prompt(context)
    result = ask_llm(final_prompt)

    if save_output:
        output_path = OUTPUT_DIR / f'{file_name}_extraction.txt'
        output_path.write_text(result, encoding='utf-8')
        print(f'Saved extraction to: {output_path}')

    return result


In [ ]:
#@title 8. Run extraction for one saved index
# Example: if your folder is batur_index, use file_name='batur'
INDEX_NAME = 'batur'

result = extract_from_index(
    file_name=INDEX_NAME,
    k=None,          # None retrieves all chunks like your original code. Try k=10 for shorter context.
    max_chars=120_000,
    save_output=True
)

print('\n========== EXTRACTION ==========\n')
print(result)


In [ ]:
#@title 9. Optional: batch run multiple saved indexes
index_names = [
    'batur_index',
    'batur-2021_index',
    'batur-2023_index',
    'batur-2025_index',
    'budenholzer_index',
    'huang_index',
    'kabokoba_index',
    'maki_index',
    'ott_index',
    'parks_index',
    'tanner_index',
    'visvanathan_index',
    'wilcken_index',
    'worringer_index',
]

batch_results = {}

for name in index_names:
    print(f'\n\n===== Processing: {name} =====')
    try:
        batch_results[name] = extract_from_index(
            file_name=name,
            k=None,
            max_chars=120_000,
            save_output=True
        )
    except Exception as e:
        batch_results[name] = f'FAILED: {e}'
        print(batch_results[name])




===== Processing: batur_index =====
Loaded: /content/drive/MyDrive/Additional_NPJ_Data/Doc_store/batur_index
Number of chunks in VDB: 11
Retrieved chunks: 11
Context characters: 66008
Saved extraction to: /content/drive/MyDrive/NPJ_RAG/zero_shot_extractions/batur_index_extraction.txt


===== Processing: batur-2021_index =====
Loaded: /content/drive/MyDrive/Additional_NPJ_Data/Doc_store/batur-2021_index
Number of chunks in VDB: 12
Retrieved chunks: 12
Context characters: 67598
Saved extraction to: /content/drive/MyDrive/NPJ_RAG/zero_shot_extractions/batur-2021_index_extraction.txt


===== Processing: batur-2023_index =====
Loaded: /content/drive/MyDrive/Additional_NPJ_Data/Doc_store/batur-2023_index
Number of chunks in VDB: 8
Retrieved chunks: 8
Context characters: 42795
Saved extraction to: /content/drive/MyDrive/NPJ_RAG/zero_shot_extractions/batur-2023_index_extraction.txt


===== Processing: batur-2025_index =====
Loaded: /content/drive/MyDrive/Additional_NPJ_Data/Doc_store/batur-2

In [ ]:
#@title 10. Optional: save all batch outputs into one JSON file
import json

combined_output_path = OUTPUT_DIR / 'all_extractions.json'
combined_output_path.write_text(
    json.dumps(batch_results, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print(f'Saved combined output to: {combined_output_path}')


## Optional improvement for future indexes: save page metadata

Your current `FAISS.from_texts(page_texts, openai_embeddings)` stores the page text but not page numbers. If you want page-aware outputs later, use metadata when creating the index:

```python
metadatas = []
page_texts = []

for page_num, items in data.items():
    page_content = ''
    # build page_content as you already do
    if page_content.strip():
        page_texts.append(page_content)
        metadatas.append({'page_num': page_num, 'source': file_name})

db_openEmbedd = FAISS.from_texts(
    texts=page_texts,
    embedding=openai_embeddings,
    metadatas=metadatas
)
```
